In [6]:
import sys
from pathlib import Path

import torch

sys.path.append("src")
from entropy_pruning import (
    AttentionForecaster,
    UNILoRAClassifier,
    build_loaders,
    evaluate_classifier,
    finetune_pruned_classifier,
    set_seed,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [11]:
CFG = dict(
    data_dir="/dune/DATASETS/NCT-CRC-HE",
    img_size=224,
    batch_size=16,
    num_workers=2,
    seed=42,
    prune_layer=0,
    target_layer=19,
    keep_ratio=0.1,
    epochs=20,
    lr_head=1e-3,
    lr_backbone=1e-4,
    weight_decay=0.01,
    label_smoothing=0.1,
)

set_seed(CFG["seed"])
dataset_name = Path(CFG["data_dir"]).name
classifier_ckpt = Path(f"checkpoints/{dataset_name}/uni_finetuned/best_model.pt")
forecaster_ckpt = Path(f"checkpoints/{dataset_name}/forecaster_ablation/forecaster_src{CFG['prune_layer']:02d}_tgt{CFG['target_layer']:02d}.pt")


In [8]:
loaders = build_loaders(
    data_dir=CFG["data_dir"],
    img_size=CFG["img_size"],
    batch_size=CFG["batch_size"],
    num_workers=CFG["num_workers"],
)
print("Classi:", loaders.class_names)


Classi: ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']


In [9]:
baseline = UNILoRAClassifier(loaders.n_classes).to(device)
baseline.load_state_dict(torch.load(classifier_ckpt, map_location=device), strict=False)
base_metrics = evaluate_classifier(baseline, loaders.test_loader, device)
print("Baseline test:", base_metrics)


Baseline test: {'acc': 0.849025069637883, 'f1_macro': 0.7882971078407449}


In [ ]:
forecaster = AttentionForecaster().to(device)
forecaster.load_state_dict(torch.load(forecaster_ckpt, map_location=device))
forecaster.eval()
for p in forecaster.parameters():
    p.requires_grad_(False)

result = finetune_pruned_classifier(
    n_classes=loaders.n_classes,
    classifier_ckpt=classifier_ckpt,
    forecaster=forecaster,
    prune_layer=CFG["prune_layer"],
    keep_ratio=CFG["keep_ratio"],
    train_loader=loaders.train_loader,
    val_loader=loaders.val_loader,
    test_loader=loaders.test_loader,
    device=device,
    epochs=CFG["epochs"],
    lr_backbone=CFG["lr_backbone"],
    lr_head=CFG["lr_head"],
    weight_decay=CFG["weight_decay"],
    label_smoothing=CFG["label_smoothing"],
)
print("Pruned result:", result)
